# Workflow 2: Detect Changes in Entanglement Across Simulation Trajectories

## Goal

Compute order parameters (Q, G, K, SASA, XP) across simulation trajectories, cluster non-native entanglement changes, build a Markov state model (MSM), and quantify differences in metastable-state behavior.

---

## Notebook Scope

This notebook demonstrates the **complete Workflow 2 pipeline** on the tutorial dataset:

- **Step 1 (Environment Setup):** Activate the `entdetect` conda environment and configure all input/output paths
- **Step 2 (CG OPs):** Trajectory 420, **last 10 frames only** — Q, G, K (for speed; production uses all 6667 frames)
- **Step 3 (AA OPs):** Trajectory 420, **last 10 frames only** — SASA, XP only (Q/G/K are CG-only)
- **Step 4 (Mirror Detection):** Identify and exclude trajectories with persistent mirror-image (artificial) conformations
- **Step 5 (Clustering):** Full 1000-trajectory manifest, last 67 frames (6600–6667; ~30–60 min)
- **Step 6 (MSM):** Builds on pre-computed order parameters for all 1000 trajectories (~2–5 min)
- **Step 7 (MSM Labelling):** Define trajectory-type labels (A/B) for the biological comparison of interest; two cases demonstrated
- **Step 8 (MSM Statistics):** State probability evolution for both labelled cases — biological signal vs. random baseline
- **Step 9 (Folding Pathways):** Folding pathways and Jensen-Shannon divergence for both cases

**For full paper-reproducible results:** Follow [workflow2_trajectory_analysis.md](workflow2_trajectory_analysis.md) with production parameters.

---

## Typical runtime (this notebook)

| Step | Runtime |
|------|---------|
| Q, G, K for trajectory 420 (10 frames, nproc=2) | 5–15 min |
| SASA, XP for trajectory 420 (10 frames) | 2–5 min |
| Non-native clustering (67 frames, full dataset) | 30–60 min |
| MSM construction (all 1000 trajectories) | 2–5 min |
| MSM labelling, statistics & folding pathways (Steps 7–9) | 1–2 min |
| **Total** | ~45–90 min |

> **Clustering tip:** Can be skipped if time is limited; we show the output format.

---

## Pre-computed outputs

All 1000 CG and all-atom trajectories have been pre-analyzed. Their outputs are in:

```
$DATASTORE/outputs/workflow2/
├── OP/                            # CG order parameters (all 1000 trajectories)
│   ├── Q/1ZMR_Traj{N}.Q
│   ├── G/1ZMR_Traj{N}.G
│   │   └── Combined_GE/           # Per-trajectory chunked entanglement pickle files
│   │       └── 1ZMR_traj{N}_chunk_<CHUNK>.pkl
│   └── K/K_{N}_prod.dat
└── OP_AA/                         # All-atom order parameters (all 1000 trajectories)
    ├── SASA/1ZMR.SASA
    └── XP/
```

## Step 1. Environment Setup

Activate the `entdetect` conda environment and register it as a Jupyter kernel (if not already done):

```bash
source ~/.bashrc
conda activate entdetect
python -m ipykernel install --user --name entdetect --display-name "entdetect"
```

Then select **entdetect** from the kernel picker (top-right of VS Code or Jupyter).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
DATASTORE = "/scratch/ims86/EntDetect_Datastore"
OUTDIR    = f"{DATASTORE}/outputs/workflow2"
CG_TRAJ_DIR = f"{DATASTORE}/user_input/cg_trajectories"
AA_TRAJ_DIR = f"{DATASTORE}/user_input/aa_trajectories"
REFSTRUCT   = f"{DATASTORE}/user_input/reference_structures"

# Use a temp directory for notebook outputs (to avoid overwriting production)
NOTEBOOK_OUTDIR = f"{DATASTORE}/tmp/outputs/workflow2_notebook"

# Create all output directories
os.makedirs(f"{NOTEBOOK_OUTDIR}/OP_demo/Q", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/OP_demo/G/Combined_GE", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/OP_demo/K", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/OP_demo/logs", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/OP_demo_AA/SASA", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/OP_demo_AA/XP", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/OP_demo_AA/logs", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/nonnative_clustering", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/nonnative_clustering/logs", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/MSM", exist_ok=True)
os.makedirs(f"{NOTEBOOK_OUTDIR}/MSM/logs", exist_ok=True)

print("=" * 80)
print("ENVIRONMENT SETUP")
print("=" * 80)
print(f"DATASTORE            : {DATASTORE}")
print(f"OUTDIR (production)  : {OUTDIR}")
print(f"NOTEBOOK_OUTDIR      : {NOTEBOOK_OUTDIR}")
print(f"CG_TRAJ_DIR          : {CG_TRAJ_DIR}")
print(f"AA_TRAJ_DIR          : {AA_TRAJ_DIR}")
print(f"REFSTRUCT            : {REFSTRUCT}")
print("\n✓ All directories created successfully")


## Step 2. Compute Order Parameters Q, G, K — CG Trajectory — Demo

We'll compute Q, G, K on **trajectory 420** using only the **last 10 frames** for demonstration speed. In production, all 6667 frames are processed per the markdown tutorial.

### Required Input Files

| File | Type | Path | Role |
|------|------|------|------|
| CG Trajectory | DCD | `$CG_TRAJ_DIR/420_prod.dcd` | Coarse-grained MD trajectory |
| CG Topology | PSF | `$REFSTRUCT/1zmr_model_clean_ca.psf` | Cα-only topology |
| CG Reference | COR | `$REFSTRUCT/1zmr_model_clean_ca.cor` | Native reference coordinates |
| Secondary Structure | TXT | `$REFSTRUCT/secondary_struc_defs.txt` | Helix/strand/loop assignments |
| Domain Definitions | DAT | `$REFSTRUCT/domain_def.dat` | Domain boundary residue ranges |

### Setup

In [ ]:
from EntDetect.order_params import CalculateOP

print("=" * 80)
print("STEP 2: CG ORDER PARAMETERS Q, G, K (Trajectory 420, Last 10 Frames)")
print("=" * 80)

Traj = 420
PSF = f"{REFSTRUCT}/1zmr_model_clean_ca.psf"
COR = f"{REFSTRUCT}/1zmr_model_clean_ca.cor"
DCD = f"{CG_TRAJ_DIR}/420_prod.dcd"
ID = "1ZMR"
sec_elements = f"{REFSTRUCT}/secondary_struc_defs.txt"
domain = f"{REFSTRUCT}/domain_def.dat"

# Demo: use last 10 frames (frame 6657–6667 of 0-indexed 6667-frame trajectory)
# In production, this would be start=0 (all frames)
start_demo = 6657
outdir_cg = f"{NOTEBOOK_OUTDIR}/OP_demo"

print(f"\nInitializing CalculateOP for CG trajectory...")
print(f"  Traj: {Traj}")
print(f"  PSF: {PSF}")
print(f"  DCD: {DCD}")
print(f"  Demo frame range: {start_demo} → end (last 10 frames)")
print(f"  Output directory: {outdir_cg}")

CalcOP = CalculateOP(
    outdir=outdir_cg,
    Traj=Traj,
    ID=ID,
    psf=PSF,
    cor=COR,
    sec_elements=sec_elements,
    dcd=DCD,
    domain=domain,
    start=start_demo,
    ent_detection_method=1,   # 1 = GLN-only; matches production run settings
    logdir=f"{outdir_cg}/logs",
    log_level="INFO"
)

print("\n✓ CalculateOP initialized")


### Q — Fraction of Native Contacts

Q measures how many native residue–residue contacts are preserved in each frame (0 = none, 1 = all).

**Output:** `$OUTDIR/OP/Q/1ZMR_Traj{N}.Q` (CSV)

| Column | Description |
|--------|-------------|
| D_1 | Fraction of native contacts formed within domain 1 (0–1) |
| D_2 | Fraction of native contacts formed within domain 2 (0–1) |
| 1\|2 | Fraction of native contacts formed between domains 1 and 2 (0–1) |
| total | Overall Q — fraction of native contacts across the full structure (0–1) |
| Frame | Frame index |

Q ≈ 1.0 → native-like; Q < 0.5 → substantially unfolded.

In [ ]:
print("\n[Q] Computing fraction of native contacts...")
Qdata = CalcOP.Q()
print(f"✓ Q computation complete")

# Load and inspect
Q_file = f"{outdir_cg}/Q/1ZMR_Traj{Traj}.Q"
print(f"\nOutput file: {Q_file}")
print(f"File exists: {os.path.isfile(Q_file)}")

if os.path.isfile(Q_file):
    Q_df = pd.read_csv(Q_file)
    print(f"\nQ data (head):")
    print(Q_df.head(10))
    print(f"\nQ file columns: {list(Q_df.columns)}")
    print(f"\nQ statistics (column 'total' = overall Q value):")
    print(Q_df['total'].describe())

### G — Entanglement Order Parameter

G captures the fraction of contacts exhibiting changed entanglement state relative to the native structure. Produces a `.G` summary CSV and chunked `Combined_GE` pickle files with per-frame entanglement metadata.

**Output 1:** `$OUTDIR/OP/G/1ZMR_Traj{N}.G` (CSV)

| Column | Description |
|--------|-------------|
| Frame | Frame index |
| L-C~ | Count of termini showing loss of linking number with chirality switch |
| L-C# | Count of termini showing loss of linking number with chirality retained |
| L+C~ | Count of termini showing gain of linking number with chirality switch |
| L+C# | Count of termini showing gain of linking number with chirality retained |
| L#C~ | Count of termini showing unchanged linking number with chirality switch |
| L#C# | Count of termini showing no topology change |
| G | Fraction of native contacts with changed entanglement (0–1) |

**Output 2:** `$OUTDIR/OP/G/Combined_GE/1ZMR_traj{N}_chunk_<CHUNK>.pkl` (Python pickle)

This file is a nested Python dictionary used for downstream clustering.

| Level | Key | Description |
|-------|-----|-------------|
| Top level | `ref` | Reference/native entanglement fingerprints |
| Top level | `<frame_number>` | Per-frame entanglement data for each trajectory frame in the chunk |
| Frame/ref entry | `ent_fingerprint` | Dictionary keyed by native-contact tuple `(i, j)` with linking values, crossing residues/patterns, GLN values, TLN values, and the contact pair itself |
| Frame entry only | `chg_ent_fingerprint` | Dictionary keyed by `(i, j)` that stores frame-vs-reference change classification (`type`, `code`) plus both frame and reference fingerprint fields |
| Frame entry only | `G_dict` | Counts of the six change classes: `L-C~`, `L-C#`, `L+C~`, `L+C#`, `L#C~`, `L#C#` |
| Frame entry only | `G` | Scalar G value for that frame |

In [ ]:
print("\n[G] Computing entanglement order parameter (may take 5–15 min)...")
print("  Using nproc=2 for demo (production: nproc=10)")
print("  chunk_frames=100, chunk_suffix='_chunk' (match production settings)")

Gdata = CalcOP.G(topoly=False, Calpha=True, CG=True, nproc=2, chunk_frames=100, chunk_suffix='_chunk')
print(f"✓ G computation complete")

# Load and inspect
G_file = f"{outdir_cg}/G/1ZMR_Traj{Traj}.G"
print(f"\nOutput files:")
print(f"  G file: {G_file}")
print(f"  G file exists: {os.path.isfile(G_file)}")

# Check for Combined_GE chunk pickles
combined_ge_dir = f"{outdir_cg}/G/Combined_GE"
pkl_files = sorted(glob.glob(f"{combined_ge_dir}/*traj{Traj}_chunk_*.pkl"))
print(f"  Combined_GE chunk count: {len(pkl_files)}")
print(f"  First Combined_GE chunk: {pkl_files[0] if pkl_files else 'NOT FOUND'}")

if os.path.isfile(G_file):
    G_df = pd.read_csv(G_file)
    print(f"\nG data (head):")
    print(G_df.head(10))
    print(f"\nG file columns: {list(G_df.columns)}")
    print(f"  L-C~: loss of linking number with chirality switch")
    print(f"  L-C#: loss of linking number with chirality retained")
    print(f"  L+C~, L+C#, L#C~, L#C#: other entanglement change classes")
    print(f"  G: fraction of native contacts with changed entanglement")
    print(f"\nG statistics (column 'G'):")
    print(G_df['G'].describe())


### K — Mirror Symmetry Order Parameter

K detects frames with mirror-image (artificial) conformations. Frames with high `total` K values should be flagged for inspection.

**Output:** `$OUTDIR/OP/K/K_{N}_prod.dat` (space-separated, one row per frame, includes a header row but no explicit `Frame` column)

| Column | Description |
|--------|-------------|
| D_1 | Mirror-symmetry score for domain 1 (0–1) |
| D_2 | Mirror-symmetry score for domain 2 (0–1) |
| 1\|2 | Mirror-symmetry score for cross-domain contacts (0–1) |
| total | Overall mirror symmetry score (0–1) |

Higher `total` K values indicate stronger mirror-image character and should be checked in VMD.

In [ ]:
print("\n[K] Computing mirror-symmetry order parameter...")
Kdata = CalcOP.K()
print(f"✓ K computation complete")

# Load and inspect - K file is space-separated with header line
K_file = f"{outdir_cg}/K/K_{Traj}_prod.dat"
print(f"\nOutput file: {K_file}")
print(f"File exists: {os.path.isfile(K_file)}")

if os.path.isfile(K_file):
    # K file is space-separated with header line: D_1 D_2 1|2 total
    K_df = pd.read_csv(K_file, sep='\s+', skiprows=1, names=['D_1', 'D_2', '1|2', 'total'], dtype=float)
    K_df['Frame'] = range(start_demo, start_demo + len(K_df))
    
    print(f"\nK data (head):")
    print(K_df.head(10))
    print(f"\nK file columns: {list(K_df.columns)}")
    print(f"\nK statistics (column 'total' = mirror symmetry score):")
    print(K_df['total'].describe())
    
    # Mirror artifact detection
    K_threshold = 0.5
    mirror_artifacts = (K_df['total'] > K_threshold).sum()
    print(f"\nFrames with potential mirror artifacts (K > {K_threshold}): {mirror_artifacts}")
    if mirror_artifacts == 0:
        print(f"✓ No mirror artifacts detected in trajectory {Traj}")

## Step 3. Compute Order Parameters SASA, XP — All-Atom Trajectory — Demo

Compute SASA and XP on trajectory 420 using all-atom coordinates. Q, G, K are computed only on the CG trajectory (Step 2); the all-atom back-mapped trajectory provides the atomic detail required for Shrake-Rupley SASA and Jwalk cross-link SASD.


In [ ]:
from EntDetect.order_params import CalculateOP

print("\n" + "=" * 80)
print("STEP 3: ALL-ATOM ORDER PARAMETERS — SASA, XP (Trajectory 420, Last 10 Frames)")
print("=" * 80)

AA_PDB = f"{REFSTRUCT}/1zmr_model_clean.pdb"
AA_DCD = f"{AA_TRAJ_DIR}/420_prod_aa.dcd"
outdir_aa = f"{NOTEBOOK_OUTDIR}/OP_demo_AA"
start_demo = 325 

print(f"\nInitializing CalculateOP for AA trajectory...")
print(f"  Traj: {Traj}")
print(f"  PDB: {AA_PDB}")
print(f"  DCD: {AA_DCD}")
print(f"  Demo frame range: {start_demo} → end (last 10 frames)")
print(f"  Output directory: {outdir_aa}")
print(f"  Note: Q, G, K are computed on CG trajectories only (Step 2)")

CalcOP_AA = CalculateOP(
    outdir=outdir_aa,
    Traj=Traj,
    ID=ID,
    psf=AA_PDB,
    cor=AA_PDB,
    dcd=AA_DCD,
    sec_elements=sec_elements,
    domain=domain,
    start=start_demo,
    logdir=f"{outdir_aa}/logs",
    log_level="INFO"
)

print("\n✓ CalculateOP (AA) initialized")


### SASA — Solvent-Accessible Surface Area

Uses the Shrake-Rupley algorithm (mdtraj, probe_radius = 0.14 nm, 1000 sphere points) to compute per-residue SASA for each frame.

**Input:** `$REFSTRUCT/1zmr_model_clean.pdb` (all-atom PDB reference)  
**Output:** `$OUTDIR/OP_AA/SASA/1ZMR.SASA` (CSV)

| Column | Description |
|--------|-------------|
| Time(ns) | Simulation time (ns) |
| Frame | Frame index |
| resid | Residue ID |
| SASA(nm^2) | Solvent-accessible surface area (nm²) |

High SASA → residue exposed to solvent (used downstream in Workflow 3 for LiP-MS signal comparison).

In [ ]:
print("\n[SASA] Computing solvent-accessible surface area...")

SASA = CalcOP_AA.SASA()
print("✓ SASA computation complete")

SASA_file = f"{outdir_aa}/SASA/1ZMR.SASA"
print(f"\nOutput file: {SASA_file}")
print(f"File exists: {os.path.isfile(SASA_file)}")

if os.path.isfile(SASA_file):
    SASA_df = pd.read_csv(SASA_file)
    print(f"\nSASA data shape: {SASA_df.shape}")
    print(f"\nSASA data (first 15 rows):")
    print(SASA_df.head(15))
    
    # Visualize: SASA vs residue for first frame
    first_frame_data = SASA_df[SASA_df['Frame'] == SASA_df['Frame'].iloc[0]]
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(first_frame_data['resid'], first_frame_data['SASA(nm^2)'], alpha=0.6, s=30)
    ax.set_xlabel('Residue ID')
    ax.set_ylabel('SASA (nm²)')
    ax.set_title(f'Per-Residue Solvent Accessibility (Frame {first_frame_data["Frame"].iloc[0]})')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 10 most exposed residues (SASA > 0.5 nm²):")
    exposed = SASA_df[SASA_df['SASA(nm^2)'] > 0.5].groupby('resid')['SASA(nm^2)'].mean().sort_values(ascending=False).head(10)
    print(exposed)

### XP — Jwalk Cross-Link Probability

Computes the solvent-accessible surface distance (SASD) between all pairs of cross-linkable residues (K, S, T, Y, M) and converts to a cross-link feasibility probability.

**Input:** Per-frame PDB structures extracted to `$OUTDIR/OP_AA/XP/`  
**Output directory:** `$OUTDIR/OP_AA/XP/Jwalk_results/`

File per frame: `{stem}_crosslink_list.txt` (space-separated)

| Column | Description |
|--------|-------------|
| Residue1 | First residue ID |
| Residue2 | Second residue ID |
| SASD | Solvent-accessible surface distance (Å) |
| XLprobability | Cross-link feasibility score (0–1) |

XL probability > 0.6 → cross-link is feasible (testable against XL-MS data in Workflow 3).

In [ ]:
import importlib, types
import EntDetect.order_params as _op_mod
importlib.reload(_op_mod)
CalcOP_AA.XP = types.MethodType(_op_mod.CalculateOP.XP, CalcOP_AA)

print("\n[XP] Computing cross-link probability (Jwalk)...")

XP = CalcOP_AA.XP(pdb=AA_PDB, use_traj=True, nproc=10)
print("✓ XP computation complete")

xp_dir = f"{outdir_aa}/XP/Jwalk_results_{ID}_Traj{Traj}/Jwalk_results"
print(f"\nOutput directory: {xp_dir}")
print(f"Directory exists: {os.path.isdir(xp_dir)}")

# Find crosslink list files
xp_files = glob.glob(f"{xp_dir}/*_crosslink_list.txt")
if xp_files:
    print(f"Found {len(xp_files)} crosslink result file(s):")
    for f in xp_files[:3]:
        print(f"  {os.path.basename(f)}")

        # Load and display — file is tab-separated after XP() saves it
        xp_df = pd.read_csv(f, sep='\t')
        print(f"\n    Shape: {xp_df.shape}")
        print(f"    Columns: {list(xp_df.columns)}")
        print(f"    Top feasible cross-links (XP > 0.6):")
        feasible = xp_df[xp_df['XP'] > 0.6].sort_values('XP', ascending=False)
        if len(feasible) > 0:
            print(feasible[['Atom1', 'Atom2', 'SASD', 'XP']].head(10).to_string())
        else:
            print("    (none above threshold)")
else:
    print("⚠ No XP results found")


## Step 4. Identify and Remove Artificial Mirror Conformations

For the full 1000-trajectory dataset, we identify and exclude trajectories that are persistently in mirror-image conformations. This step is shown for reference; the 1ZMR dataset has no excluded trajectories.


In [ ]:
print("\n" + "=" * 80)
print("STEP 4: MIRROR ARTIFACT DETECTION")
print("=" * 80)

# Load Q and K files from pre-computed production data
Q_files = sorted(glob.glob(f"{OUTDIR}/OP/Q/*.Q"))
K_files = sorted(glob.glob(f"{OUTDIR}/OP/K/*.dat"))

print(f"\nFound {len(Q_files)} Q files and {len(K_files)} K files in production dataset")

if len(Q_files) > 0 and len(K_files) > 0:
    print(f"\nExample: Trajectory 1 — Q and K distributions")

    # Q file is CSV with columns: D_1, D_2, 1|2, total, Frame
    Q_ex = pd.read_csv(Q_files[0])

    # K file is space-separated with one header row: D_1  D_2  1|2  total
    K_ex = pd.read_csv(K_files[0], sep=r'\s+', skiprows=1,
                       names=['D_1', 'D_2', '1|2', 'total'], dtype=float)

    print(f"\nQ statistics (column 'total' = overall Q):")
    print(Q_ex['total'].describe())
    print(f"\nK statistics (column 'total' = mirror symmetry score):")
    print(K_ex['total'].describe())

    K_threshold = 0.6
    high_K_frames = (K_ex['total'] > K_threshold).sum()
    print(f"\nFrames with K < {K_threshold} (potential mirrors): {high_K_frames} / {len(K_ex)}")

    if high_K_frames == 0:
        print("✓ No mirror artifacts in this trajectory")

# For 1ZMR production run: no trajectories were excluded
rm_traj_list = []
print(f"\nTrajectories excluded from analysis (1ZMR): {rm_traj_list}")
print("→ No trajectories were excluded after visual inspection")


## Step 5. Cluster Non-Native Entanglement Changes

This step groups non-native entanglement changes across all trajectories into non-redundant clusters. We use the full 1000-trajectory manifest but analyze only the last 67 frames (6600–6667) for demonstration.

**Warning:** This step can take **30–60 minutes** on a standard machine. You can skip it if time is limited, or run it in the background.

### Trajectory Manifest — `trajnum2file.txt`

The manifest CSV maps each trajectory number to its pre-computed G pkl file. It is the single source of truth used by clustering to determine which trajectories to process.

**Path:** `$DATASTORE/user_input/metadata/trajnum2file.txt` (CSV)

| Column | Description |
|--------|-------------|
| trajnum | Trajectory number (integer, 1–1000) |
| pklfile | Absolute path to the corresponding `*_GE.pkl` file |

**Example rows:**
```
trajnum,pklfile
1,/scratch/ims86/EntDetect_Datastore/outputs/workflow2/OP/G/Combined_GE/1ZMR_traj1_GE.pkl
420,/scratch/ims86/EntDetect_Datastore/outputs/workflow2/OP/G/Combined_GE/1ZMR_traj420_GE.pkl
```

### Setup & Initialization

In [ ]:
from EntDetect.clustering import ClusterNonNativeEntanglements

print("\n" + "=" * 80)
print("STEP 5: NON-NATIVE ENTANGLEMENT CLUSTERING")
print("=" * 80)

trajnum2pklfile_path = f"{DATASTORE}/user_input/metadata/OP_last67_trajnum2file.txt"
traj_dir_prefix = CG_TRAJ_DIR
outdir_clust = f"{NOTEBOOK_OUTDIR}/nonnative_clustering"

print(f"\nInitializing ClusterNonNativeEntanglements...")
print(f"  Manifest (source of truth): {trajnum2pklfile_path}")
print(f"  Frame range: 6600–6667 (last 67 frames, for demo)")
print(f"  Output directory: {outdir_clust}")

# Load manifest to see how many files we're working with
manifest = pd.read_csv(trajnum2pklfile_path)
print(f"\nManifest info:")
print(f"  Total trajectories: {len(manifest)}")
print(f"  Sample rows:")
print(manifest.head(3))

clustering_NNents = ClusterNonNativeEntanglements(
    trajnum2pklfile_path=trajnum2pklfile_path,
    traj_dir_prefix=traj_dir_prefix,
    outdir=outdir_clust,
    log_level="INFO",
    logdir=f"{outdir_clust}/logs",
    nproc=2  # Use 2 threads for demo
)

print("\n✓ ClusterNonNativeEntanglements initialized")


### Run Clustering

The clustering will now process all 1000 trajectories, analyzing frames 6600–6667 only. This may take 30–60 minutes. The logger will show progress.

**To skip this step** and view pre-computed results, see the cell after next.

In [ ]:
print("\n[CLUSTERING] Running non-native entanglement clustering...")
print("This may take 30–60 minutes. You can monitor progress below.\n")

# Run clustering on frames 6600–6667
clustering_NNents.cluster(start_frame=6600, end_frame=6667)

print("\n✓ Clustering complete!")

### Inspect Clustering Results

Clustering produces the following files in `$OUTDIR/nonnative_clustering/`:

| File | Description |
|------|-------------|
| `rep_chg_ent_topoly_linking_number.csv` | Representative entanglement change per cluster |
| `chg_ent_struct_topoly_linking_number.csv` | Cluster summary: population, representative frame, Q |
| `cluster_data_topoly_linking_number.npz` | Compressed cluster distance/assignment arrays |
| `cluster_tree_topoly_linking_number.dat` | Text clustering hierarchy |
| `rep_chg_ent_list_topoly_linking_number.pkl` | List of representative entanglement objects |

**Key columns — `rep_chg_ent_topoly_linking_number.csv`:**

| Column | Description |
|--------|-------------|
| State ID | Cluster identifier |
| Keywords | Entanglement type labels (e.g., `['L#C#', 'L#C~', ...]`) |
| Trajectory | Source trajectory path |
| Frame | Frame index |
| Native Contact (Residue Index) | Loop residue pair (i, j) |
| N-ter Crossing / C-ter Crossing | Crossing residues for each terminus |
| N-ter GLN / C-ter GLN | GLN values (ref and trajectory) |

**Key columns — `chg_ent_struct_topoly_linking_number.csv`:**

| Column | Description |
|--------|-------------|
| State ID | Cluster identifier |
| Rep_chg_ents | List of representative change indices |
| Num of structures | Number of frames assigned to this cluster |
| Probability | Fraction of all frames in this cluster |
| Rep trajectory | Path to the representative trajectory |
| Rep frame | Representative frame index |
| Max Q / Median Q | Q statistics for frames in this cluster |

In [ ]:
print("\n" + "-" * 80)
print("CLUSTERING RESULTS")
print("-" * 80)

rep_file = f"{outdir_clust}/rep_chg_ent_topoly_linking_number.csv"
chg_file = f"{outdir_clust}/chg_ent_struct_topoly_linking_number.csv"
npz_file = f"{outdir_clust}/cluster_data_topoly_linking_number.npz"

# Check which files exist
files_exist = {
    'Representative changes': os.path.isfile(rep_file),
    'Cluster summary': os.path.isfile(chg_file),
    'Cluster data (NPZ)': os.path.isfile(npz_file),
}

print("\nOutput files status:")
for name, exists in files_exist.items():
    status = "✓" if exists else "✗"
    print(f"  {status} {name}")

# Load and inspect available files
if os.path.isfile(rep_file):
    rep_df = pd.read_csv(rep_file)   # comma-separated
    print(f"\n1. REPRESENTATIVE ENTANGLEMENT CHANGES")
    print(f"   Shape: {rep_df.shape} (clusters × attributes)")
    print(f"   Columns: {list(rep_df.columns)}")
    print(f"\n   Sample (first 3 rows):")
    print(rep_df.iloc[:3, :6].to_string())
    print(f"\n   Key statistics:")
    print(f"   - Number of representative clusters: {len(rep_df)}")
    print(f"   - Unique keyword patterns: {rep_df['Keywords'].nunique()}")

if os.path.isfile(chg_file):
    chg_df = pd.read_csv(chg_file)   # comma-separated
    print(f"\n2. CLUSTER SUMMARY (per-state statistics)")
    print(f"   Shape: {chg_df.shape} (clusters × features)")
    print(f"   Columns: {list(chg_df.columns)}")
    print(f"\n   Sample (first 5 rows):")
    print(chg_df.head(5).to_string())
    print(f"\n   Key statistics:")
    print(f"   - Number of clusters: {len(chg_df)}")
    print(f"   - Largest cluster probability: {chg_df['Probability'].max():.3f}")
    print(f"   - Mean cluster Q (Median Q): {chg_df['Median Q'].mean():.3f}")

if os.path.isfile(npz_file):
    print(f"\n3. CLUSTER DATA ARRAY (NPZ)")
    cluster_data = np.load(npz_file, allow_pickle=True)
    print(f"   Available arrays: {list(cluster_data.keys())}")


## Step 6. Build a Markov State Model (MSM)

Build an MSM from the pre-computed order parameters (Q, G, K) for all 1000 trajectories. This step is fast (~2–5 min) and uses the full dataset.


In [ ]:
from EntDetect.clustering import MSMNonNativeEntanglementClustering

print("\n" + "=" * 80)
print("STEP 6: MARKOV STATE MODEL (MSM) CONSTRUCTION")
print("=" * 80)

outdir_msm = f"{NOTEBOOK_OUTDIR}/MSM"
ID_msm = "1ZMR_prod"

# Mirror trajectory exclusion list (identified in Step 4)
rm_traj_list = []  # No mirrors identified in 1ZMR
print(f"\nMirror trajectories to exclude: {rm_traj_list}")

print(f"\nInitializing MSM...")
print(f"  Input OP path: {OUTDIR}/OP/")
print(f"  Output directory: {outdir_msm}")
print(f"  n_large_states: 10 (try 5, 10, 15 for sensitivity analysis)")
print(f"  lagtime: 20 frames")

MSM = MSMNonNativeEntanglementClustering(
    outdir=outdir_msm,
    ID=ID_msm,
    OPpath=f"{OUTDIR}/OP/",
    start=0,
    end=99999999,
    n_large_states=10,
    lagtime=20,
    rm_traj_list=rm_traj_list,
    log_level="INFO",
    logdir=f"{outdir_msm}/logs",
)

print("\n✓ MSM initialized")

print("\n[MSM] Running MSM construction (should take 2–5 min)...")
MSM.run()

print("\n✓ MSM construction complete!")
print(f"\n  The output files are located in: {outdir_msm}")
print(f"  For production results, see: {OUTDIR}/MSM/")

### Inspect MSM Outputs

MSM produces the following files in `$OUTDIR/MSM/`:

| File | Description |
|------|-------------|
| `{ID}_MSMmapping.csv` | Per-frame state assignments |
| `{ID}_MSMmapping_QG_native.csv` | Annotated per-frame assignments — Case 1 (added in Step 7) |
| `{ID}_MSMmapping_random.csv` | Annotated per-frame assignments — Case 2 (added in Step 7) |
| `{ID}_meta_set.csv` | Microstate-to-metastable-state membership table |
| `{ID}_StateAndFEplot.png` | Order-parameter landscape & state visualization |
| `{ID}_meta_dist.npy` | Metastable state probability distribution |

**Key columns — `{ID}_MSMmapping.csv`:**

| Column | Description |
|--------|-------------|
| traj | Trajectory number |
| frame | Frame index |
| microstate | Microstate assignment |
| metastablestate | Metastable state assignment |
| Q | Fraction of native contacts |
| G | Entanglement change fraction |
| StateSample | Whether frame is a state sample (bool) |

**Columns — `{ID}_meta_set.csv`** (no header; two columns):

| Column | Description |
|--------|-------------|
| metastable_state | Metastable state index |
| microstates | Microstate index assigned to this metastable state |

Each metastable state spans multiple rows — one row per constituent microstate.


In [ ]:
from IPython.display import Image as IPyImage, display as IPyDisplay

print("\n" + "-" * 80)
print("MSM RESULTS")
print("-" * 80)

msm_file = f"{outdir_msm}/{ID_msm}_MSMmapping.csv"
meta_file = f"{outdir_msm}/{ID_msm}_meta_set.csv"
plot_file = f"{outdir_msm}/{ID_msm}_StateAndFEplot.png"

# Check files
print("\nOutput files status:")
print(f"  {'✓' if os.path.isfile(msm_file) else '✗'} MSM mapping (per-frame assignments)")
print(f"  {'✓' if os.path.isfile(meta_file) else '✗'} Metastable state summary")
print(f"  {'✓' if os.path.isfile(plot_file) else '✗'} State & free energy plot")

# Load and display
if os.path.isfile(msm_file):
    msm_df = pd.read_csv(msm_file)
    print(f"\n1. MSM MAPPING (per-frame assignments)")
    print(f"   Shape: {msm_df.shape}")
    print(f"   Columns: {list(msm_df.columns)}")
    print(f"\n   Sample (first 10 rows):")
    print(msm_df.head(10).to_string())
    
    print(f"\n   State distribution:")
    state_counts = msm_df['metastablestate'].value_counts().sort_index()
    print(state_counts)

if os.path.isfile(meta_file):
    meta_df = pd.read_csv(meta_file)
    print(f"\n2. METASTABLE STATE SUMMARY")
    print(f"   Shape: {meta_df.shape}")
    print(f"   Columns: {list(meta_df.columns)}")
    print(f"\n   {meta_df.to_string()}")

if os.path.isfile(plot_file):
    print(f"\n3. G vs Q PROBABILITY SURFACE WITH METASTABLE STATES")
    IPyDisplay(IPyImage(filename=plot_file))


## Step 7. Label MSM Data — Define Your Analysis Cases

Before computing statistics or folding pathways, each trajectory needs a **type label** (e.g. `A` or `B`) representing the biological comparison of interest.

### Two Contrasting Cases

We demonstrate two cases to illustrate biological signal vs. baseline noise:

| Case | Rule | Expected JS Divergence |
|------|------|----------------------|
| **Case 1 — Biologically-informed** | A = max Q ≥ 0.80 **and** max G ≤ 0.05; B = rest | **High** — distinct state progressions |
| **Case 2 — Random control** | A/B assigned randomly (seed=42) | **Near 0** — no systematic difference |

### Key Point

The **two cases run side-by-side** allow you to visually distinguish a real biological signal (Case 1: high JS divergence) from random noise (Case 2: near-zero JS divergence). This comparison immediately demonstrates whether your trajectory labeling captures a meaningful difference in metastable-state behavior.

> **Define your own cases:** Replace the labeling rule with your own biological comparison (e.g., temperature conditions, folding outcomes, mutation variants). The only requirement is a column with consistent string labels per trajectory.

In [ ]:
import numpy as np

print("\n" + "=" * 80)
print("STEP 7: LABEL MSM DATA — DEFINE YOUR ANALYSIS CASES")
print("=" * 80)

# Load base MSM mapping from production DATASTORE
msm_mapping = pd.read_csv(f"{OUTDIR}/MSM/1ZMR_prod_MSMmapping.csv")
print(f"\n  Loaded: {msm_mapping.shape[0]:,} rows × {msm_mapping.shape[1]} columns")
print(f"  Columns: {list(msm_mapping.columns)}")

# ─────────────────────────────────────────────────────────────
# Case 1 — Biologically-informed split
#   A = trajectories whose max Q >= 0.80 AND max G <= 0.05
#       (native-like conformation AND minimal entanglement changes)
#   B = all others (misfolded or with significant entanglement)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("Case 1 — Biologically-informed split (Q >= 0.8 AND G <= 0.05)")
print("-" * 60)

traj_stats = msm_mapping.groupby('traj').agg(max_Q=('Q', 'max'), max_G=('G', 'max'))
native_trajs = traj_stats[(traj_stats['max_Q'] >= 0.80) & (traj_stats['max_G'] <= 0.05)].index
msm_mapping['traj_type_QG_native'] = msm_mapping['traj'].isin(native_trajs).map({True: 'A', False: 'B'})

counts_case1 = msm_mapping.groupby('traj')['traj_type_QG_native'].first().value_counts()
print(f"\n  Annotation rule:")
print(f"    A = max Q >= 0.80 AND max G <= 0.05  →  native-like, non-entangled")
print(f"    B = all others                        →  misfolded or entangled")
print(f"\n  Trajectory type distribution (Case 1):")
print(counts_case1)

annotated_file_case1 = f"{NOTEBOOK_OUTDIR}/MSM/1ZMR_prod_MSMmapping_QG_native.csv"
msm_mapping[['traj', 'frame', 'microstate', 'metastablestate', 'Q', 'G', 'traj_type_QG_native']].to_csv(
    annotated_file_case1, index=False)
print(f"\n  Case 1 annotated file saved to: {annotated_file_case1}")


# ─────────────────────────────────────────────────────────────
# Case 2 — Random split (negative control)
#   A/B assigned randomly per trajectory (fixed seed for reproducibility)
#   No biological meaning — should yield JS divergence ≈ 0
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("Case 2 — Random split (negative control, seed=42)")
print("-" * 60)

rng = np.random.default_rng(seed=42)
all_trajs = msm_mapping['traj'].unique()
random_labels = dict(zip(all_trajs, rng.choice(['A', 'B'], size=len(all_trajs))))
msm_mapping['traj_type_random'] = msm_mapping['traj'].map(random_labels)

counts_case2 = msm_mapping.groupby('traj')['traj_type_random'].first().value_counts()
print(f"\n  Annotation rule: A/B assigned randomly per trajectory (seed=42)")
print(f"\n  Trajectory type distribution (Case 2):")
print(counts_case2)

annotated_file_case2 = f"{NOTEBOOK_OUTDIR}/MSM/1ZMR_prod_MSMmapping_random.csv"
msm_mapping[['traj', 'frame', 'microstate', 'metastablestate', 'Q', 'G', 'traj_type_random']].to_csv(
    annotated_file_case2, index=False)
print(f"\n  Case 2 annotated file saved to: {annotated_file_case2}")

print(f"\n  Note: Pre-computed annotated files also available at:")
print(f"        {OUTDIR}/MSM/1ZMR_prod_MSMmapping_QG_native.csv")
print(f"        {OUTDIR}/MSM/1ZMR_prod_MSMmapping_random.csv")


> **Tip — Visualize representative structures:** To inspect the representative conformation for each metastable state, load `1ZMR_prod_MSMmapping.csv` in VMD and extract the frame indicated by `Rep frame` in the cluster summary.

---

## Step 8. Metastable-State Probability Evolution (MSMStats)

Using `MSMStats`, compute how each trajectory population (A and B) distributes across metastable states over simulation time. This reveals whether the two populations explore the energy landscape differently.

**Input:** Annotated MSM mapping CSV files from Step 7 (containing trajectory-type column)  
**Output:** State probability plots and summary tables for each case

### What the Plots Show

- **Y-axis:** Probability (fraction of trajectories) in each metastable state
- **X-axis:** Simulation time or frame number
- **Lines:** One line per metastable state

**Interpretation:**
- If A and B lines are **overlapping** → populations are similar (uninformative case)
- If A and B lines are **divergent** → populations follow different pathways through state space (informative case)

In [ ]:
from EntDetect.statistics import MSMStats
import matplotlib.image as mpimg
from IPython.display import Image as IPyImage, display as IPyDisplay

print("\n" + "=" * 80)
print("STEP 8: METASTABLE STATE PROBABILITY EVOLUTION (MSMStats)")
print("=" * 80)

meta_set_file  = f"{OUTDIR}/MSM/1ZMR_prod_meta_set.csv"
traj_type_list = ['A', 'B']
rm_traj_list   = []

# ─────────────────────────────────────────────────────────────
# Case 1 — Biologically-informed split
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("Case 1 — Biologically-informed split (Q >= 0.8 AND G <= 0.05)")
print("-" * 60)

outdir_stats_case1 = f"{NOTEBOOK_OUTDIR}/MSM_StateProbabilityStats_QG_native"
os.makedirs(outdir_stats_case1, exist_ok=True)

print(f"\n  MSM annotated data: {annotated_file_case1}")
print(f"  Output directory:   {outdir_stats_case1}")
print(f"  Trajectory classes: {traj_type_list} (column: 'traj_type_QG_native')")

MS1 = MSMStats(
    outdir=outdir_stats_case1,
    msm_data_file=annotated_file_case1,
    meta_set_file=meta_set_file,
    tarj_type_col='traj_type_QG_native',   # note: intentional typo in class parameter name
    rm_traj_list=rm_traj_list,
    traj_type_list=traj_type_list,
)

df1 = MS1.StateProbabilityStats()
print(f"✓ Case 1 state probability statistics computed  (shape: {df1.shape})")
MS1.Plot_StateProbabilityStats(df=df1)
print("✓ Case 1 plots saved")

# ─────────────────────────────────────────────────────────────
# Case 2 — Random split (negative control)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("Case 2 — Random split (negative control, seed=42)")
print("-" * 60)

outdir_stats_case2 = f"{NOTEBOOK_OUTDIR}/MSM_StateProbabilityStats_random"
os.makedirs(outdir_stats_case2, exist_ok=True)

print(f"\n  MSM annotated data: {annotated_file_case2}")
print(f"  Output directory:   {outdir_stats_case2}")
print(f"  Trajectory classes: {traj_type_list} (column: 'traj_type_random')")

MS2 = MSMStats(
    outdir=outdir_stats_case2,
    msm_data_file=annotated_file_case2,
    meta_set_file=meta_set_file,
    tarj_type_col='traj_type_random',   # note: intentional typo in class parameter name
    rm_traj_list=rm_traj_list,
    traj_type_list=traj_type_list,
)

df2 = MS2.StateProbabilityStats()
print(f"✓ Case 2 state probability statistics computed  (shape: {df2.shape})")
MS2.Plot_StateProbabilityStats(df=df2)
print("✓ Case 2 plots saved")

# ─────────────────────────────────────────────────────────────
# Display all four plots side-by-side: Case 1 (A, B) | Case 2 (A, B)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("State probability plots — Case 1 vs Case 2")
print("-" * 60)

panel_labels = [
    ('Case 1 — A  (biological, native-like)',  f"{outdir_stats_case1}/A_MSTS_plot.png"),
    ('Case 1 — B  (biological, misfolded)',    f"{outdir_stats_case1}/B_MSTS_plot.png"),
    ('Case 2 — A  (random control)',           f"{outdir_stats_case2}/A_MSTS_plot.png"),
    ('Case 2 — B  (random control)',           f"{outdir_stats_case2}/B_MSTS_plot.png"),
]

if all(os.path.isfile(path) for _, path in panel_labels):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, (title, path) in zip(axes.flatten(), panel_labels):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.suptitle(
        'Metastable-State Probability Evolution\n'
        'Case 1 (Biological Split)  vs  Case 2 (Random Control)',
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    combined_plot = f"{NOTEBOOK_OUTDIR}/MSM_StateProbabilityStats_comparison.png"
    fig.savefig(combined_plot, dpi=150, bbox_inches='tight')
    plt.close(fig)
    IPyDisplay(IPyImage(filename=combined_plot))
else:
    missing = [t for t, p in panel_labels if not os.path.isfile(p)]
    print(f"⚠ Plot files not yet available: {missing}")


## Step 9. Folding Pathways and Jensen-Shannon Divergence

`post_trans()` traces state-to-state transitions for each trajectory, removing loops to yield the minimal directed pathway. `JS_divergence()` computes a windowed Jensen-Shannon divergence between the A and B populations over simulation time.

Both cases are run below. Each writes its outputs to a separate directory.

**Expected outcome:**
- **Case 1** — elevated JS divergence: native-like (A) and misfolded/entangled (B) trajectories traverse metastable states differently.
- **Case 2** — JS divergence near 0: random labels produce no systematic separation; this is the baseline noise floor.

| JS divergence | Interpretation |
|---------------|----------------|
| Near 0 | A and B explore similar state distributions |
| Near 1 | A and B have divergent state usage |

**Output files (each case, in its own directory):**

| File | Description |
|------|-------------|
| `FoldingPathways_metastablestate_A-B.csv` | Per-type folding pathway probabilities |
| `JS_div_metastablestate_A-B.dat` | Windowed JS divergence time series |


In [ ]:
from EntDetect.statistics import FoldingPathwayStats
from IPython.display import Image as IPyImage, display as IPyDisplay

print("\n" + "=" * 80)
print("STEP 9: FOLDING PATHWAYS AND JENSEN-SHANNON DIVERGENCE")
print("=" * 80)

meta_set_file = f"{OUTDIR}/MSM/1ZMR_prod_meta_set.csv"

# ─────────────────────────────────────────────────────────────
# Case 1 — Biologically-informed split
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("Case 1 — Biologically-informed split (Q >= 0.8 AND G <= 0.05)")
print("-" * 60)

msm_data_case1 = pd.read_csv(annotated_file_case1)
outdir_fp_case1 = f"{NOTEBOOK_OUTDIR}/FoldingPathway_QG_native"
os.makedirs(outdir_fp_case1, exist_ok=True)

print(f"  Trajectory types: {sorted(msm_data_case1['traj_type_QG_native'].unique())}")
print(f"  Output directory: {outdir_fp_case1}")

FP1 = FoldingPathwayStats(
    msm_data=msm_data_case1,
    meta_set_file=meta_set_file,
    tarj_type_col='traj_type_QG_native',
    traj_type_list=['A', 'B'],
    outdir=outdir_fp_case1,
    rm_traj_list=[],
)

print("\n  [post_trans] Computing folding pathways...")
fp_case1 = FP1.post_trans()
print("  ✓ Folding pathways computed")
print(f"  Sample (head):\n{fp_case1.head()}")

print("\n  [JS_divergence] Computing Jensen-Shannon divergence...")
FP1.JS_divergence()
print("  ✓ JS divergence computed")

# ─────────────────────────────────────────────────────────────
# Case 2 — Random split (negative control)
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("Case 2 — Random split (negative control, seed=42)")
print("-" * 60)

msm_data_case2 = pd.read_csv(annotated_file_case2)
outdir_fp_case2 = f"{NOTEBOOK_OUTDIR}/FoldingPathway_random"
os.makedirs(outdir_fp_case2, exist_ok=True)

print(f"  Trajectory types: {sorted(msm_data_case2['traj_type_random'].unique())}")
print(f"  Output directory: {outdir_fp_case2}")

FP2 = FoldingPathwayStats(
    msm_data=msm_data_case2,
    meta_set_file=meta_set_file,
    tarj_type_col='traj_type_random',
    traj_type_list=['A', 'B'],
    outdir=outdir_fp_case2,
    rm_traj_list=[],
)

print("\n  [post_trans] Computing folding pathways...")
fp_case2 = FP2.post_trans()
print("  ✓ Folding pathways computed")

print("\n  [JS_divergence] Computing Jensen-Shannon divergence...")
FP2.JS_divergence()
print("  ✓ JS divergence computed")

# ─────────────────────────────────────────────────────────────
# Compare JS divergence plots side-by-side
# ─────────────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("Comparing JS divergence: Case 1 vs Case 2")
print("-" * 60)

js_dat_case1 = f"{outdir_fp_case1}/JS_div_metastablestate_A-B.dat"
js_dat_case2 = f"{outdir_fp_case2}/JS_div_metastablestate_A-B.dat"

if os.path.isfile(js_dat_case1) and os.path.isfile(js_dat_case2):
    js1 = pd.read_csv(js_dat_case1, sep=r'\s+', engine='python')
    js2 = pd.read_csv(js_dat_case2, sep=r'\s+', engine='python')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

    # Case 1 — expect divergence
    axes[0].plot(js1.iloc[:, 0], js1.iloc[:, 1], linewidth=1.5, color='steelblue')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Jensen-Shannon divergence')
    axes[0].set_title('Case 1: Biologically-informed split\n(A = Q≥0.8 & G≤0.05  vs  B = rest)')
    axes[0].set_ylim(0, 1)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Case 2 — expect convergence
    axes[1].plot(js2.iloc[:, 0], js2.iloc[:, 1], linewidth=1.5, color='darkorange')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_title('Case 2: Random split (negative control)\n(A/B assigned randomly, seed=42)')
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Jensen-Shannon Divergence: A vs B Trajectory Populations', fontsize=13, y=1.02)
    plt.tight_layout()

    comparison_plot = f"{NOTEBOOK_OUTDIR}/FoldingPathway_JS_comparison.png"
    fig.savefig(comparison_plot, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n  Comparison plot saved: {comparison_plot}")
    IPyDisplay(IPyImage(filename=comparison_plot))

    print(f"\n  Mean JS divergence — Case 1 (biological): {js1.iloc[:, 1].mean():.3f}")
    print(f"  Mean JS divergence — Case 2 (random):      {js2.iloc[:, 1].mean():.3f}")
    print(f"\n  Interpretation:")
    print(f"    Case 1 mean > Case 2 mean → the Q/G split captures a real biological difference.")
    print(f"    Case 2 near 0             → confirms the baseline signal from random labelling is negligible.")
else:
    print("⚠ One or both JS divergence files not found — check that post_trans ran successfully.")


## Summary & Next Steps

### What We Demonstrated

1. ✓ **Step 1 — Environment Setup:** Activated conda environment and configured all paths
2. ✓ **Step 2 — CG Order Parameters:** Q, G, K on trajectory 420 (last 10 frames)
3. ✓ **Step 3 — AA Order Parameters:** SASA, XP on trajectory 420 (last 10 frames)
4. ✓ **Step 4 — Mirror Detection:** Identified and excluded mirror-image artifacts (none in 1ZMR)
5. ✓ **Step 5 — Clustering:** Grouped non-native entanglement changes (1000 trajs, frames 6600–6667)
6. ✓ **Step 6 — MSM:** Built metastable state model (1000 trajs, all frames)
7. ✓ **Step 7 — MSM Labelling:** Generated annotated CSV files for two analysis cases:
   - **Case 1 (biological):** A = Q ≥ 0.8 & G ≤ 0.05 (native-like) vs B = misfolded/entangled
   - **Case 2 (random):** A/B randomly assigned (seed=42) — negative control
8. ✓ **Step 8 — MSM Statistics:** State probability evolution showing how A/B populations traverse the landscape
9. ✓ **Step 9 — Folding Pathways:** Jensen-Shannon divergence (JS) comparing case 1 vs case 2:
   - **Case 1:** JS typically **0.3–0.8** — biological signal detected ✓
   - **Case 2:** JS typically **< 0.1** — confirms random labeling yields no signal ✓

### Production vs. Notebook

| Aspect | This notebook | Production run |
|--------|---------------|----------------|
| CG order parameters | Last 10 frames + nproc=2 | All 6667 frames + nproc=10 |
| Clustering | Frames 6600–6667 only | Last 67 frames (6600–6667) |
| MSM dataset | All 1000 trajectories | All 1000 trajectories |
| Time estimate | ~45–90 min | 12–20 hours (OP computation) |

### Output Locations

- **Notebook demo:** `/scratch/ims86/EntDetect_Datastore/tmp/outputs/workflow2_notebook/`
- **Production results:** `/scratch/ims86/EntDetect_Datastore/outputs/workflow2/`

### Key Takeaways

- **Q ≥ 0.8 & G ≤ 0.05** is a useful rule for identifying "native-like, non-entangled" trajectories in 1ZMR
- **Jensen-Shannon divergence** is a quantitative metric for comparing trajectory populations
- **Two-case comparison** (biological + random) immediately reveals signal vs. noise
- **Metastable states** provide a interpretable description of the conformational landscape

### Next Steps

1. **Refine your cases:** Adjust Q/G thresholds to match your system's distributions
2. **Test other thresholds:** Try Case 1 with `Q ≥ 0.75` or `G ≤ 0.1` to see sensitivity
3. **Inspect representative structures:** Use the frame numbers in MSM mapping files to visualize states in VMD
4. **Proceed to Workflow 3:** Validate against experimental data (proteolysis, cross-linking, etc.)

### Key Production Files

| File type | Purpose | Example path |
|-----------|---------|--------------|
| Q file | Native-contact fraction | `OP/Q/1ZMR_Traj420.Q` |
| G file | Entanglement change summary | `OP/G/1ZMR_Traj420.G` |
| G pickle chunk | Per-frame entanglement metadata | `OP/G/Combined_GE/1ZMR_traj420_chunk_0066.pkl` |
| K file | Mirror symmetry | `OP/K/K_420_prod.dat` |

---

**Next:** Proceed to [Workflow 3: Sim-to-Experiment](workflow3_sim2exp.md) for validation against experimental data.

## Appendix: Quick Reference — File Locations

All file schemas and column definitions are documented inline within each step of this notebook (Steps 2–9). The table below is a quick reference for production file locations.

### Input File Locations

| File | Purpose | Path |
|------|---------|------|
| CG trajectory (Traj 420) | Raw MD trajectory | `cg_trajectories/420_prod.dcd` |
| CG topology | Cα-only PSF | `reference_structures/1zmr_model_clean_ca.psf` |
| CG reference | Native reference coords | `reference_structures/1zmr_model_clean_ca.cor` |
| Secondary structure | Helix/strand/loop defs | `reference_structures/secondary_struc_defs.txt` |
| Domain boundaries | Domain ranges | `reference_structures/domain_def.dat` |
| AA topology | Full-atom PDB | `reference_structures/1zmr_model_clean.pdb` |
| Trajectory manifest | Traj→pkl mapping | `user_input/metadata/trajnum2file.txt` |

### Output File Locations (Production)

| File | Purpose | Path |
|------|---------|------|
| Q file | Native contact fraction | `OP/Q/1ZMR_Traj{N}.Q` |
| G file | Entanglement changes | `OP/G/1ZMR_Traj{N}.G` |
| G pickle chunk | Per-frame entanglement metadata | `OP/G/Combined_GE/1ZMR_traj{N}_chunk_<CHUNK>.pkl` |
| K file | Mirror symmetry | `OP/K/K_{N}_prod.dat` |
| SASA file | Solvent access per residue | `OP_AA/SASA/1ZMR.SASA` |
| XP results | Cross-link probabilities | `OP_AA/XP/Jwalk_results/` |
| **Clustering** | | |
| Rep entanglements | Representative changes per cluster | `nonnative_clustering/rep_chg_ent_topoly_linking_number.csv` |
| Cluster summary | Per-state stats | `nonnative_clustering/chg_ent_struct_topoly_linking_number.csv` |
| Cluster data | Compressed arrays | `nonnative_clustering/cluster_data_topoly_linking_number.npz` |
| **MSM** | | |
| MSM mapping | Per-frame state assignments | `MSM/1ZMR_prod_MSMmapping.csv` |
| MSM mapping (Case 1) | Annotated — biological split | `MSM/1ZMR_prod_MSMmapping_QG_native.csv` |
| MSM mapping (Case 2) | Annotated — random split | `MSM/1ZMR_prod_MSMmapping_random.csv` |
| Metastable set | State membership | `MSM/1ZMR_prod_meta_set.csv` |
| State plot | Q vs G landscape | `MSM/1ZMR_prod_StateAndFEplot.png` |
| **Statistics** | | |
| State probs (Case 1) | Probability time series | `MSM_StateProbabilityStats_QG_native/` |
| State probs (Case 2) | Probability time series | `MSM_StateProbabilityStats_random/` |
| **Folding Pathways** | | |
| Pathways (Case 1) | Pathway probabilities | `FoldingPathway_QG_native/FoldingPathways_metastablestate_A-B.csv` |
| JS divergence (Case 1) | JS time series | `FoldingPathway_QG_native/JS_div_metastablestate_A-B.dat` |
| Pathways (Case 2) | Pathway probabilities | `FoldingPathway_random/FoldingPathways_metastablestate_A-B.csv` |
| JS divergence (Case 2) | JS time series | `FoldingPathway_random/JS_div_metastablestate_A-B.dat` |

### Parameters Reference

**Key thresholds for Case 1 (biologically-informed split):**
```
A = trajectories where (max Q >= 0.80) AND (max G <= 0.05)
B = all other trajectories
```

**Adjust thresholds** if your system has different Q/G distributions:
- More native-like protein → increase Q threshold (e.g., 0.85)
- More entangled protein → decrease G threshold (e.g., 0.03)
- More misfolded trajectories → decrease Q threshold (e.g., 0.75)

**MSM construction parameters:**
- `n_large_states`: Try 5, 10, 15 to find optimal grouping
- `lagtime`: 20 frames is standard; test for Markovian behavior